# Hafta 13 — Derin Öğrenmeye Bakış

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy torch torchvision pillow transformers

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Çok katmanlı sinir ağlarına **derin öğrenme** denir. Görüntü tanıyan bir ağda ilk katmanlar çizgileri ve kenarları tanımayı öğrenir. Orta katmanlar göz, kulak gibi parçaları, son katmanlar ise kedinin tamamını tanır.

Bu tür ağları sıfırdan eğitmek için milyonlarca örnek gerekir. Bu yüzden çoğu zaman çok büyük bir veri setiyle eğitilmiş hazır bir model alınır ve az sayıda örnekle yeni bir göreve uyarlanır. Bisiklet sürmeyi bilen birinin motosiklet kullanmayı daha kısa sürede öğrenmesi de benzer bir durumdur.

Sohbet robotlarının temelinde ise Transformer adlı bir yapı bulunur. Bu yapı, bir cümledeki her kelimenin diğer kelimelerle olan ilişkisine "dikkat ederek" çalışır.

---

## Bu Haftanın Hedefleri

- Evrişim ve havuzlama işlemlerini elle hesaplamak
- Bir CNN'in görüntülerden nasıl öznitelik öğrendiğini anlamak
- Transfer öğrenme ile az veriyle model eğitmek
- Dikkat mekanizmasının temel fikrini kavramak

---

## 1. Evrişimli Sinir Ağları (CNN)

### Neden özel bir mimari?

224×224 piksellik renkli bir görüntü 150.528 sayıdan oluşur. Bu görüntüyü sıradan bir ağa verip ilk katmana 1.000 nöron koyarsak yalnızca bu katmanda **150 milyon** ağırlık olur. Ayrıca ağ, sol üst köşedeki kediyi tanımayı öğrense bile sağ alttaki kediyi ayrıca öğrenmek zorunda kalır.

CNN bu sorunları iki fikirle çözer:
- **Yerel bakış:** Her nöron görüntünün yalnızca küçük bir bölgesine (ör. 3×3) bakar.
- **Ağırlık paylaşımı:** Aynı küçük filtre görüntünün her yerinde kullanılır. Böylece bir desen nerede olursa olsun tanınır.

### Evrişim işlemi

Küçük bir **filtre**, görüntünün üzerinde kaydırılır. Her konumda filtre ile altındaki görüntü parçası eleman eleman çarpılıp toplanır.

**Elle örnek — dikey kenar tespiti:** Görüntünün sol yarısı parlak (10), sağ yarısı karanlık (0).

```
Görüntü (6×6):              Filtre (3×3):        Sonuç (4×4):
10 10 10  0  0  0           1  0  −1             0  30  30  0
10 10 10  0  0  0           1  0  −1             0  30  30  0
10 10 10  0  0  0           1  0  −1             0  30  30  0
10 10 10  0  0  0                                0  30  30  0
10 10 10  0  0  0
10 10 10  0  0  0
```

- Sol üst konumda her yer 10 olduğu için sonuç 10 + 10 + 10 − 10 − 10 − 10 = **0** olur; bu bölgede değişim yoktur.
- Bir sağa kayınca filtrenin sağ sütunu karanlık bölgeye denk gelir: 10 + 10 + 10 − 0 − 0 − 0 = **30** olur; burada kenar vardır.

Sonuçtaki büyük değerler kenarın yerini gösterir. CNN'de bu filtreler elle tasarlanmaz; eğitim sırasında **veriden öğrenilir**.

In [ ]:
import numpy as np

def evrisim(goruntu, filtre):
    k = filtre.shape[0]
    H, W = goruntu.shape[0] - k + 1, goruntu.shape[1] - k + 1
    return np.array([[np.sum(goruntu[i:i + k, j:j + k] * filtre) for j in range(W)] for i in range(H)])

goruntu = np.array([[10, 10, 10, 0, 0, 0]] * 6)
filtre = np.array([[1, 0, -1]] * 3)
print(evrisim(goruntu, filtre))

**Çıktı boyutu** = (girdi − filtre + 2·dolgu) / adım + 1. Örneğin 6×6 girdi ve 3×3 filtreyle dolgu olmadan sonuç 4×4'tür.

### Havuzlama

Havuzlama, görüntüyü küçültür. **Maksimum havuzlama** her 2×2'lik bloğun en büyük değerini alır:

```
1 3 | 2 1
4 6 | 5 0          6  5
────┼────    →
3 1 | 1 2          3  4
0 2 | 4 3
```

Bu işlem hesaplamayı azaltır ve küçük kaymalara karşı dayanıklılık sağlar.

### Tipik CNN yapısı

```
Görüntü → [Evrişim → ReLU → Havuzlama] × birkaç kez → Düzleştir → Tam bağlı katman → Sınıf
```

Derin katmanlara gidildikçe görüntü küçülür, filtre sayısı artar ve öğrenilen desenler kenarlardan dokulara, oradan da nesne parçalarına doğru soyutlaşır.

In [ ]:
import torch
import torch.nn as nn

cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28×28 → 14×14
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14×14 → 7×7
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10),
)
print(cnn(torch.randn(8, 1, 28, 28)).shape)       # torch.Size([8, 10])

---

## 2. Transfer Öğrenme

Milyonlarca görüntüyle (ImageNet) eğitilmiş bir ağ, kenar, doku ve şekil gibi genel görsel özellikleri zaten öğrenmiştir. Kendi küçük veri setimiz için sıfırdan başlamak yerine bu bilgiyi yeniden kullanırız.

| Strateji | Ne yapılır? | Ne zaman? |
|---|---|---|
| **Öznitelik çıkarma** | Hazır ağın katmanları dondurulur, yalnızca son katman yeniden eğitilir | Veri çok azsa |
| **İnce ayar** | Son katmanlar da küçük bir öğrenme oranıyla eğitilir | Veri biraz daha fazlaysa |

**Veri artırma:** Eğitim görüntüleri rastgele çevrilir, döndürülür veya kırpılır. Böylece az veriden daha fazla çeşitlilik elde edilir.

---

## 3. Dizi Modelleri ve Transformer

Metin ve konuşma gibi verilerde **sıra önemlidir**: "Köpek adamı ısırdı" ile "Adam köpeği ısırdı" aynı şey değildir.

- **RNN:** Cümleyi kelime kelime okur ve o ana kadar okuduklarının bir özetini taşır. Uzun cümlelerde baştaki bilgiyi unutma eğilimindedir.
- **LSTM:** Bilgiyi daha uzun süre saklayabilen "kapılar" ekleyerek bu sorunu azaltır.
- **Transformer:** Kelimeleri sırayla okumak yerine her kelimenin **diğer bütün kelimelere doğrudan bakmasını** sağlar. Bu mekanizmaya **dikkat (attention)** denir.

> "Hayvan sokağı geçmedi çünkü **o** çok yorgundu." cümlesinde "o" kelimesi en çok "hayvan" kelimesine dikkat etmelidir.

### Dikkat mekanizması: Küçük bir örnek

Her kelime bir sayı vektörüyle temsil edilir. Bir kelimenin yeni temsili, diğer kelimelerin kendisine ne kadar benzediğine göre ağırlıklandırılmış bir ortalamadır.

Üç kelime: x₁ = (1, 0), x₂ = (0, 1), x₃ = (1, 1). Birinci kelime için:

```
1) Benzerlik (iç çarpım):   x₁·x₁ = 1,  x₁·x₂ = 0,  x₁·x₃ = 1
2) Ölçekle (÷ √2) ve softmax ile ağırlıklara çevir:   (0.40, 0.20, 0.40)
3) Ağırlıklı ortalama:   0.40·(1,0) + 0.20·(0,1) + 0.40·(1,1) = (0.80, 0.60)
```

Birinci kelime en çok kendisine ve ona benzeyen üçüncü kelimeye dikkat etti.

In [ ]:
def dikkat(X):
    skor = X @ X.T / np.sqrt(X.shape[1])
    agirlik = np.exp(skor) / np.exp(skor).sum(axis=1, keepdims=True)    # softmax
    return agirlik @ X, agirlik

cikti, agirlik = dikkat(np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]))
print(agirlik.round(2)); print(cikti.round(2))

Transformer'lar bütün kelimeleri aynı anda işlediği için çok hızlı eğitilir. BERT ve GPT gibi büyük dil modellerinin temelinde bu yapı vardır.

---

## 4. Laboratuvar: Hazır Bir CNN ile Görüntü Sınıflandırma

Her sınıftan yaklaşık 120 eğitim görüntüsüyle karınca ve arı fotoğraflarını sınıflandıracağız.

In [ ]:
import os, zipfile, urllib.request
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

if not os.path.exists("hymenoptera_data"):
    urllib.request.urlretrieve("https://download.pytorch.org/tutorial/hymenoptera_data.zip", "veri.zip")
    zipfile.ZipFile("veri.zip").extractall(".")

donusum = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                              transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
egitim = DataLoader(datasets.ImageFolder("hymenoptera_data/train", donusum), batch_size=16, shuffle=True)
dogr = DataLoader(datasets.ImageFolder("hymenoptera_data/val", donusum), batch_size=16)

# Öznitelik çıkarma: hazır ResNet-18'i dondur, yalnızca son katmanı eğit
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)             # yeni çıktı katmanı: karınca / arı

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
kayip_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for X, y in egitim:
        optimizer.zero_grad(); kayip_fn(model(X), y).backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        dogru = sum((model(X).argmax(1) == y).sum().item() for X, y in dogr)
    print(f"Epoch {epoch + 1}: doğrulama doğruluğu = {dogru / len(dogr.dataset):.3f}")   # ~%90+

**Görevler:**
1. `weights=None` ile aynı modeli sıfırdan eğitin ve doğruluğu karşılaştırın. Fark neden bu kadar büyük?
2. Kendi telefonunuzla iki sınıftan 30'ar fotoğraf çekip (`veri/train/sinif_adi/` klasör yapısıyla) aynı kodu çalıştırın.

---

## 5. Özet

- **Evrişim**, küçük bir filtreyi görüntü üzerinde kaydırarak desenleri bulur; filtreler veriden öğrenilir.
- **Havuzlama** görüntüyü küçültür ve küçük kaymalara dayanıklılık sağlar.
- **Transfer öğrenme**, hazır bir ağın bilgisini kullanarak az veriyle yüksek başarı sağlar.
- **Transformer**, dikkat mekanizmasıyla her kelimenin diğer bütün kelimelere bakmasını sağlar.

## Kendinizi Sınayın

1. 32×32 bir görüntüye dolgusuz 5×5 filtre uygulanırsa çıktı kaç × kaç olur?
2. Havuzlama katmanının öğrenilen bir ağırlığı var mıdır?
3. 100 görüntülük bir veri setinde transfer öğrenmenin hangi stratejisini seçersiniz?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.